In [ ]:
from thbsplines.hierarchical_space import HierarchicalSpace
import numpy as np
import dolfinx
import basix.ufl
import pyvista
from mpi4py import MPI


from dolfinx import default_real_type, default_scalar_type
rtype = default_real_type
dtype = default_scalar_type
import ufl


from thbsplines.refinement import refine
from thbsplines.fenicsx.mesh import build_mesh
from thbsplines.fenicsx.functionspace import build_dofmap, fill_function_space, create_spline_space
from thbsplines.fenicsx.solvers import solve_problem
from thbsplines.fenicsx.adaptivity import dorfler_marking
from thbsplines.fenicsx.kernels import make_linear_kernel, make_bilinear_kernel
from thbsplines.fenicsx.postprocessing import map_spline_to_legendre
from thbsplines.fenicsx.forms import mark_cells, make_bilinear_form, make_linear_form

In [ ]:
p0 = 2
m=3
n_refinements = 0
knots1 = np.array([-1,0.,1], dtype=np.float64)
knots1 = refine(knots1, p=p0, n_times=n_refinements)
log_initial_mesh_size = np.log2(np.max(np.diff(knots1)))
knots2 =refine(np.array([-1,0, 1.], dtype=np.float64), p0, n_times=n_refinements)
err_cells = {}
hs = HierarchicalSpace(knots=[knots1, knots2], degrees=[p0])

In [ ]:
for level, cells in err_cells.items():
    hs.refine(cells, level, refine_neighbours=False, refine_T_neighbours=True, m=m)
hs.hmesh.plot_cells()

disconnected_mesh, thb_operators, N_max, _ = build_mesh(hs=hs)

In [ ]:
legendre_elt = basix.ufl.element(
    "DG",
    "quadrilateral",
    degree=p0,
    lagrange_variant=basix.LagrangeVariant.legendre
)
V = dolfinx.fem.functionspace(disconnected_mesh, legendre_elt)
print(f"Number of degrees of freedom: {V.dofmap.index_map.size_global}")
dx_custom = ufl.Measure("dx", domain=disconnected_mesh, metadata={"quadrature_degree": 12})
u,v = ufl.TrialFunction(V), ufl.TestFunction(V) 
my_x = ufl.SpatialCoordinate(disconnected_mesh)
# f = dolfinx.fem.Function(V)
# f.interpolate(lambda x: (np.tanh(9*x[1]-9*x[0])+1)/9. + 1./(1.5*np.exp((10.*x[0]-6.)**2 + 
#                                                                        (10.*x[1]+7)**2)) + 
#                                                                        1./(np.exp(np.sqrt((2*x[0]+1)**2 + (2*x[1]-1)**2))))
#f = my_x[0]*my_x[1]*(knots1[-1]-my_x[0])*(knots2[-1]-my_x[1])**2
sigma = 0.4
#f = (1./(3.141592*sigma**4))*(1.-0.5*((my_x[0]**2+my_x[1]**2)/sigma**2))*ufl.exp(-(my_x[0]**2+my_x[1]**2)/(2.*sigma**2))
#f = (ufl.tanh(9.*(my_x[1]-my_x[0]))+1)/9. + 2./3*ufl.exp(-ufl.sqrt((10.*my_x[0]-6.)**2+(10.*my_x[1]+7.)**2))
f = (ufl.tanh(9.*(my_x[1]-my_x[0]))+1)/9. + (2./3.)*ufl.exp(-(10.*my_x[0]-6.)**2-(10.*my_x[1]+7.)**2)
a0 = ufl.inner(u, v) * dx_custom
f0 = ufl.inner(f, v)*dx_custom

f_square_integral = dolfinx.fem.assemble_scalar(dolfinx.fem.form(ufl.inner(f,f)*dx_custom))
f_sq_integral = np.sqrt(disconnected_mesh.comm.allreduce(f_square_integral, op=MPI.SUM))

In [ ]:
dofmap, padded_cells_to_dofs = build_dofmap(hierarchical_space=hs, mesh=disconnected_mesh, 
                                            N_max=N_max, morton=True)
C_func, C_space = fill_function_space(hierachical_space=hs, mesh=disconnected_mesh,
                                      N_max=N_max, thb_operators=thb_operators)
V_spline = create_spline_space(cells_to_dofs=padded_cells_to_dofs, mesh=disconnected_mesh,
                               N_max=N_max, mult_factor=1)

local_dofs = (hs.degrees[0]+1)**2
tabulate_A = make_bilinear_kernel(disconnected_mesh, a0, padded_dofs=N_max, local_dofs=local_dofs)
tabulate_b = make_linear_kernel(disconnected_mesh, f0, padded_dofs=N_max, local_dofs=local_dofs)

cell_domain = mark_cells(mesh=disconnected_mesh)
a_cond = make_bilinear_form(mesh=disconnected_mesh,
                            ufl_form=a0,
                            trial_space=V_spline, test_space=V_spline,
                            coefficients=C_func, 
                            integrals=[(cell_domain, tabulate_A)])
l_cond = make_linear_form(mesh=disconnected_mesh, 
                          ufl_form=f0,
                          test_space=V_spline,
                          coefficients=C_func,
                          integrals=[(cell_domain, tabulate_b)])

In [ ]:
x_vec, A = solve_problem(hs=hs, a=a_cond, rhs=l_cond, dirichlet_indices=None, 
                      dummy_index=np.max(padded_cells_to_dofs), V_spline=V_spline,
                      iterative=False, return_A=True)

u_dg = map_spline_to_legendre(hs, V, C_func, N_max, disconnected_mesh, padded_cells_to_dofs, x_vec)
u_dg.x.scatter_forward()


# Compute exact L2 error using FEniCSx standard UFL
error_form = dolfinx.fem.form(ufl.inner(f - u_dg, f - u_dg) * dx_custom)
error_sq = dolfinx.fem.assemble_scalar(error_form)
exact_l2_error = np.sqrt(disconnected_mesh.comm.allreduce(error_sq, op=MPI.SUM))

print(f"Exact L2 Error (via DG projection): {exact_l2_error:.4e}")
rel_err = exact_l2_error/f_sq_integral
print(f"Relative error = {rel_err:.4e}\n")

V_error = dolfinx.fem.functionspace(disconnected_mesh, ("DG", 0))
v = ufl.TestFunction(V_error)
local_error_form = dolfinx.fem.form(ufl.inner(f - u_dg, f - u_dg) * v * dx_custom)

err_cells = dorfler_marking(hierarchical_space=hs, theta=0.7, local_error_form=local_error_form)

In [ ]:
# import dolfinx.plot
# import pyvista

# #1. Create a "Nodal" DG space of the same degree for plotting
# #By default, DG with no variant specified uses Lagrange (nodal)
# v_plot_elt = basix.ufl.element(
#     "DG", 
#     "quadrilateral", 
#     degree=p0+2
# )
# V_plot = dolfinx.fem.functionspace(disconnected_mesh, v_plot_elt)

# # 2. Interpolate your computed solution (u_dg) into the nodal space
# u_plot = dolfinx.fem.Function(V_plot)
# #error_ufl = ufl.ln(ufl.sqrt((u_dg-f)**2)+1e-5)
# error_ufl = u_dg
# error_expr = dolfinx.fem.Expression(error_ufl, V_plot.element.interpolation_points)
# u_error = dolfinx.fem.Function(V_plot)
# u_error.interpolate(error_expr)

# # 3. Now use V_plot for the VTK mesh generation
# topology, cell_types, geometry = dolfinx.plot.vtk_mesh(V_plot)
# grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# # 4. Attach the interpolated values
# grid.point_data["u"] = u_error.x.array.real
# grid.set_active_scalars("u")

# # 5. Plotting (with a 'shrink' to see your disconnected mesh boundaries!)
# plotter = pyvista.Plotter()
# grid_shrink = grid.shrink(0.95) # This makes the "disconnected" nature visible
# plotter.add_mesh(grid_shrink, show_edges=False, cmap="turbo")
# plotter.view_xy()
# plotter.show(jupyter_backend="static")
# plotter.show()

# from pyvista.trame.jupyter import launch_server
# pyvista.set_jupyter_backend('client')
# warped_grid = grid.warp_by_scalar("u", factor=1.) 

# # If you still want to see the gaps between cells:
# grid_shrink = warped_grid.shrink(0.95)

# plotter.add_mesh(grid_shrink, show_edges=False, cmap="viridis", lighting=True)

# # Set a nice 3D camera angle instead of view_xy()
# plotter.camera_position = 'iso' 
# await launch_server().ready
# plotter.show()